# PySpark reference 12: Python UDFs, pandas and Arrow

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-12-python-udfs-and-pandas.ipynb)

Python UDFs and their pitfalls, Arrow-optimized UDFs, pandas UDFs (scalar, iterator, grouped aggregate), `applyInPandas`, `mapInPandas`, `cogroup`, Python UDTFs, Arrow conversion, and when to use built-ins instead. Practise it in [PySpark practice 12: Python UDFs, pandas and Arrow](pyspark-practice-12-python-udfs-and-pandas.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

from typing import Iterator

import pandas as pd

## Contents

| section | entries |
| --- | --- |
| [Python UDFs](#python-udfs) | [`F.udf(f=None, returnType=StringType(), useArrow=None)`](#fudffnone-returntypestringtype-usearrownone), [`spark.udf.register(name, f, returnType=None)`](#sparkudfregistername-f-returntypenone), [Arrow-optimized Python UDFs](#arrow-optimized-python-udfs) |
| [pandas UDFs](#pandas-udfs) | [`F.pandas_udf(f=None, returnType=None)`: Series to Series](#fpandas_udffnone-returntypenone-series-to-series), [Iterator pandas UDFs](#iterator-pandas-udfs), [Grouped aggregate pandas UDFs](#grouped-aggregate-pandas-udfs) |
| [pandas function APIs](#pandas-function-apis) | [`groupBy(...).applyInPandas(func, schema)`](#groupbyapplyinpandasfunc-schema), [`mapInPandas(func, schema)` and `mapInArrow`](#mapinpandasfunc-schema-and-mapinarrow), [`groupBy(...).cogroup(other.groupBy(...)).applyInPandas(func, schema)`](#groupbycogroupothergroupbyapplyinpandasfunc-schema) |
| [Python UDTFs](#python-udtfs) | [`@F.udtf(returnType=...)`: table functions](#fudtfreturntype-table-functions) |
| [Arrow and pandas conversion](#arrow-and-pandas-conversion) | [`toPandas`, `createDataFrame(pandas_df)` and Arrow](#topandas-createdataframepandas_df-and-arrow) |
| [Choosing between UDFs and built-ins](#choosing-between-udfs-and-built-ins) | [Cost and alternatives](#cost-and-alternatives) |

## Python UDFs

### `F.udf(f=None, returnType=StringType(), useArrow=None)`

Wraps a Python function so Spark calls it **once per row** on the executors. Rows are serialized from the JVM to a
Python worker process, processed, and serialized back.

| parameter | type | description |
| --- | --- | --- |
| `f` | function | the Python function; also usable as a decorator `@F.udf("type")` |
| `returnType` | DDL string or `DataType`, default string | the declared result type |
| `useArrow` | bool, optional | transfer data with Arrow (faster); default from `spark.sql.execution.pythonUDF.arrow.enabled` |

**Returns:** a function that takes Columns and returns a Column.

**Notes**
- The function receives `None` for null inputs; handle it, or the UDF raises inside the task.
- If the function returns a value that does not match `returnType` (an `int` for `"string"`, a `float` for `"int"`),
  the classic UDF returns **null silently**. Arrow UDFs are stricter.
- Spark cannot optimize through a UDF: no predicate pushdown, no constant folding, and the plan shows
  `BatchEvalPython` (or `ArrowEvalPython`). Use a built-in function whenever one exists.
- `.asNondeterministic()` marks a UDF whose output can change between calls (random values), so the optimizer does not
  reorder or deduplicate it.

In [ ]:
@F.udf("string")
def mask_name(name):
    if name is None:
        return None
    return name[0] + "*" * (len(name) - 1)

@F.udf("int")
def wrong_type(amount):
    return float(amount) * 2  # returns a float for an int column: the classic UDF turns it into null

customers.select("name", mask_name("name").alias("masked")).show(3)
orders.select("amount", wrong_type("amount").alias("silently_null")).show(2)
print([l.strip()[:60] for l in plan_lines(customers.select(mask_name("name"))) if "EvalPython" in l])

### `spark.udf.register(name, f, returnType=None)`

Registers a Python function (or an existing UDF) under a name usable in SQL and `F.expr`.

**Returns:** the UDF, also usable from Python.

In [ ]:
spark.udf.register("mask_name", mask_name)
spark.sql("SELECT name, mask_name(name) AS masked FROM {c} LIMIT 2", c=customers).show()

### Arrow-optimized Python UDFs

With `useArrow=True` (or `spark.sql.execution.pythonUDF.arrow.enabled=true`), a regular row-at-a-time UDF transfers
data in Arrow batches instead of pickled rows, which is usually much faster for simple functions. The function is
still called per value; only the transport changes. The plan shows `ArrowEvalPython`.

**Notes:** Arrow UDFs coerce return values more strictly: a type mismatch raises instead of returning null.

In [ ]:
arrow_mask = F.udf(lambda s: None if s is None else s.upper(), "string", useArrow=True)
print([l.strip()[:60] for l in plan_lines(customers.select(arrow_mask("name"))) if "EvalPython" in l])
customers.select(arrow_mask("name").alias("upper")).show(2)

## pandas UDFs

### `F.pandas_udf(f=None, returnType=None)`: Series to Series

A vectorized UDF: Spark sends **batches** of rows through Arrow and the function receives whole `pandas.Series`
(one per argument) and returns a Series of the same length. The type hints decide the kind of pandas UDF.

| type hints | kind | used in |
| --- | --- | --- |
| `pd.Series, ... -> pd.Series` | scalar | `select`, `withColumn` |
| `Iterator[pd.Series] -> Iterator[pd.Series]` | scalar iterator | `select`; setup once per partition |
| `Iterator[Tuple[pd.Series, ...]] -> Iterator[pd.Series]` | multi-column iterator | `select` |
| `pd.Series, ... -> scalar` | grouped aggregate | `groupBy().agg()`, windows |

**Notes:** batch size is `spark.sql.execution.arrow.maxRecordsPerBatch` (10,000). Decimals arrive as Python
`Decimal` objects in an `object` Series; convert deliberately. pandas UDFs need `pandas` and `pyarrow` on the workers.

In [ ]:
@F.pandas_udf("double")
def zscore_in_batch(amount: pd.Series) -> pd.Series:
    values = amount.astype(float)
    return (values - values.mean()) / values.std()

@F.pandas_udf("string")
def domain(email: pd.Series) -> pd.Series:
    return email.str.split("@").str[-1].str.lower()

orders.select("order_id", "amount", F.round(zscore_in_batch("amount"), 2).alias("z_within_batch")).show(4)
spark.createDataFrame([("ada@Example.com",), ("grace@corp.io",)], "email string").select(domain("email").alias("domain")).show()

### Iterator pandas UDFs

`Iterator[pd.Series] -> Iterator[pd.Series]` receives all batches of a partition through one iterator, so expensive
setup (loading a model, opening a connection, compiling a regex) runs **once per partition** instead of once per batch.

In [ ]:
@F.pandas_udf("double")
def with_rate(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    rate = 1.2  # imagine loading a rate table or a model here, once per partition
    for amounts in batches:
        yield amounts.astype(float) * rate

orders.select("order_id", "amount", with_rate("amount").alias("gross")).show(3)

### Grouped aggregate pandas UDFs

`pd.Series -> scalar` aggregates each group (or window frame) in pandas. Use it for statistics Spark lacks, such as a
weighted median or a custom robust mean.

**Notes:** all values of a group are loaded into one Python worker's memory; very large groups can exhaust it.

In [ ]:
@F.pandas_udf("double")
def weighted_avg(price: pd.Series, qty: pd.Series) -> float:
    price = price.astype(float)
    return float((price * qty).sum() / qty.sum())

order_items.join(products, "sku").groupBy("category").agg(F.round(weighted_avg("unit_price", "qty"), 2).alias("weighted_avg_price")).orderBy("category").show()

## pandas function APIs

### `groupBy(...).applyInPandas(func, schema)`

Runs `func(pdf: pandas.DataFrame) -> pandas.DataFrame` once **per group** with all of the group's rows, and stacks the
results. The output can have any number of rows and any columns, declared in `schema`.

| parameter | type | description |
| --- | --- | --- |
| `func` | function | `pd.DataFrame -> pd.DataFrame`, or `(key, pd.DataFrame) -> pd.DataFrame` to receive the group key |
| `schema` | DDL string or `StructType` | the output schema; columns are matched by name |

**Use it for:** per-group logic that is natural in pandas: fitting a model per store, interpolation, custom resampling.

**Notes:** the whole group must fit in one Python worker's memory; this is the main risk with skewed keys.

In [ ]:
def normalize(pdf: pd.DataFrame) -> pd.DataFrame:
    amount = pdf["amount"].astype(float)
    pdf["share"] = (amount / amount.sum()).round(3)
    return pdf[["customer_id", "order_id", "share"]]

orders.groupBy("customer_id").applyInPandas(normalize, "customer_id int, order_id int, share double").orderBy("order_id").show(5)

### `mapInPandas(func, schema)` and `mapInArrow`

`df.mapInPandas(func, schema)` passes each partition to `func(iterator of pandas DataFrames)` and expects an iterator of
pandas DataFrames back, of any length. `mapInArrow` does the same with `pyarrow.RecordBatch` objects, avoiding the pandas
conversion.

**Use it for:** row-wise transformations that need a Python library and may change the number of rows (parsing,
filtering, calling an SDK in batches).

In [ ]:
def big_orders(batches):
    for pdf in batches:
        yield pdf[pdf["amount"].astype(float) > 200][["order_id", "amount"]]

orders.mapInPandas(big_orders, "order_id int, amount decimal(12,2)").orderBy("order_id").show()

### `groupBy(...).cogroup(other.groupBy(...)).applyInPandas(func, schema)`

Groups two DataFrames by the same key and calls `func(left_pdf, right_pdf)` once per key with both groups. Use it for
per-key logic over two datasets, such as an as-of merge with `pandas.merge_asof`.

In [ ]:
rates = spark.createDataFrame([(1, datetime(2026, 1, 5, 9, 0), 1.1), (1, datetime(2026, 1, 5, 11, 0), 1.2), (2, datetime(2026, 1, 6, 0, 0), 1.3)],
                              "customer_id int, ts timestamp, rate double")

def asof(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    merged = pd.merge_asof(left.sort_values("order_ts"), right.sort_values("ts"), left_on="order_ts", right_on="ts", direction="backward")
    return merged[["order_id", "rate"]]

(orders.where("customer_id <= 2").groupBy("customer_id")
 .cogroup(rates.groupBy("customer_id")).applyInPandas(asof, "order_id int, rate double").orderBy("order_id").show())

## Python UDTFs

### `@F.udtf(returnType=...)`: table functions

A Python **user-defined table function** returns a table instead of a value: a class with an `eval(self, *args)` method
that `yield`s output rows (tuples). Call it in Python with `MyUdtf(F.lit(...))`, or register it with
`spark.udtf.register(name, cls)` and use it in SQL `FROM` or with `LATERAL` to expand each input row.

**Use it for:** exploding a value into rows with custom logic (tokenizing text, expanding date ranges with business
rules, calling an API that returns several records).

In [ ]:
@F.udtf(returnType="word string, position int")
class Tokens:
    def eval(self, text: str):
        for i, word in enumerate((text or "").lower().split(), 1):
            yield word, i

Tokens(F.lit("Fast data Engineering")).show()
spark.udtf.register("tokens", Tokens)
spark.sql("SELECT n.name, t.word FROM {c} n, LATERAL tokens(n.name) t WHERE n.customer_id <= 2 ORDER BY 1, t.position", c=customers).show()

## Arrow and pandas conversion

### `toPandas`, `createDataFrame(pandas_df)` and Arrow

With `spark.sql.execution.arrow.pyspark.enabled=true`, `toPandas()` and `spark.createDataFrame(pdf)` move data through
Apache Arrow in columnar batches, typically 10–100× faster than row-by-row conversion. If a type is not supported,
Spark falls back to the slow path (unless `...arrow.pyspark.fallback.enabled=false`).

| setting | purpose |
| --- | --- |
| `spark.sql.execution.arrow.pyspark.enabled` | Arrow for `toPandas` and `createDataFrame(pandas)` |
| `spark.sql.execution.arrow.maxRecordsPerBatch` | rows per Arrow batch (memory per batch) |
| `spark.sql.execution.pythonUDF.arrow.enabled` | Arrow transport for regular Python UDFs |

**Notes:** pandas extension dtypes (such as `StringDtype`) and Python `Decimal` objects can make Arrow fall back to
the slow path with a warning; convert to plain `object`, `float` or `int` columns first. `toPandas()` still collects
everything to the driver. For pandas-style code at scale, the pandas API on Spark
(`import pyspark.pandas as ps`) runs pandas syntax distributed.

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
pdf = orders.select("order_id", "status", "amount").toPandas()
print(type(pdf).__name__, pdf.shape, pdf.dtypes.to_dict())
back = spark.createDataFrame(pdf.assign(amount=pdf["amount"].astype(float), status=pdf["status"].astype(object)))  # plain types keep Arrow on
print(back.dtypes)

## Choosing between UDFs and built-ins

### Cost and alternatives

| approach | relative speed | use when |
| --- | --- | --- |
| built-in functions, SQL expressions, higher-order functions | fastest; optimized and code-generated | almost always |
| SQL UDF (`CREATE FUNCTION ... RETURN expr`) | as fast as built-ins (inlined) | reusable expressions |
| pandas UDF / Arrow UDF | fast for vectorizable logic | numeric or string logic that needs Python libraries |
| `applyInPandas` / `mapInPandas` | fast, but whole groups or batches in Python memory | per-group or per-batch Python logic |
| classic Python UDF | slowest: per-row serialization | small data or no alternative |

The plan below shows the difference: the built-in is code-generated inside one stage and the filter is applied
directly, while the UDF adds `BatchEvalPython` steps that ship rows to Python. Here it runs **twice**, once for the
filter and once for the projection, because Spark cannot reuse an opaque function's result.

**Checklist before writing a UDF:** search `pyspark.sql.functions` (string, regex, date, JSON, array and map functions,
higher-order `transform`/`filter`/`aggregate`), then SQL UDFs, then pandas UDFs.

In [ ]:
big = spark.range(1000).select((F.col("id") % 10).cast("string").alias("s"))
py_upper = F.udf(lambda s: s.upper(), "string")
for label, expr in [("built-in upper", F.upper("s")), ("python udf", py_upper("s"))]:
    lines = plan_lines(big.select(expr.alias("u")).where(F.col("u") == "5"))
    print(f"{label}:")
    for line in lines:
        if any(k in line for k in ("Project", "EvalPython", "Filter")):
            print("   ", line.strip()[:90])

**Next:** practise these entries in [PySpark practice 12: Python UDFs, pandas and Arrow](pyspark-practice-12-python-udfs-and-pandas.ipynb).